In [1]:
RUN_MODE = "observed-dev"
CONTRACT_VERSION = "2.1.2"
CRAWL_RELEASE_ID = "CRAWL_20260806_03"
DATA_VERSION = "observed-dev-20260806.1"
AS_OF_DATE = "2026-08-06"
RANDOM_SEED = 42
DATA_PROVENANCE = "OBSERVED_DEVELOPMENT_ONLY"
EMPIRICAL_ANALYSIS_ALLOWED = False
PROMOTION_ALLOWED = False
DUTY_INPUT_PATH = ""
GOLD_INPUT_PATH = ""
CONTROL_SCHEMA_DIR = ""

# P4 Agent 4 · Observed Duty Mapping

**Stage:** `A4-03-MAP-OBSERVED` · **Mode:** `observed-dev` · **Contract:** `2.1.2`

Validate 28 duty rows and materialize lexical top-5 candidates with unmapped preservation.

> Development-only orchestration. Empirical analysis and production promotion are disabled.

In [2]:
from pathlib import Path
import os
import sys
import pandas as pd

NCS_ROOT = Path.cwd().resolve()
if NCS_ROOT.name != 'ncs_mapping':
    raise RuntimeError('run this notebook with cwd=ncs_mapping')
sys.path.insert(0, str(NCS_ROOT / 'src'))
assert RUN_MODE == 'observed-dev'
assert DATA_PROVENANCE == 'OBSERVED_DEVELOPMENT_ONLY'
assert EMPIRICAL_ANALYSIS_ALLOWED is False and PROMOTION_ALLOWED is False
resolved_duty_input = DUTY_INPUT_PATH or os.environ.get('P4_A2_DUTY_HANDOFF', '')
resolved_gold_input = GOLD_INPUT_PATH or os.environ.get('P4_NCS_GOLD_INPUT', '')
resolved_schema_dir = CONTROL_SCHEMA_DIR or os.environ.get('P4_CONTROL_SCHEMA_DIR', '')

In [3]:
from p4_ncs.contracts.observed_duty import load_and_validate_observed_duties
from p4_ncs.dictionary.alias_dictionary import load_alias_dictionary
from p4_ncs.mapping.observed_baseline import map_observed_duties
from p4_ncs.retrieval.lexical_index import LexicalIndex

if not resolved_duty_input:
    raise FileNotFoundError('DUTY_INPUT_PATH or P4_A2_DUTY_HANDOFF is required')
duties, duty_validation, duty_envelope = load_and_validate_observed_duties(resolved_duty_input)
ncs_units = pd.read_parquet(NCS_ROOT / 'data/processed/ncsUnit.parquet')
codeset = pd.read_parquet(NCS_ROOT / 'data/processed/coreAiItCodeSet.parquet')
aliases = load_alias_dictionary(NCS_ROOT / 'configs/ncs_alias_dictionary.yaml')
candidates_preview, matches_preview = map_observed_duties(duties, LexicalIndex.build(ncs_units, codeset), aliases, codeset, DATA_VERSION, top_k=5)
input_audit = {'dutyRows': duty_validation.row_count, 'candidateRows': len(candidates_preview), 'matchRows': len(matches_preview), 'maxTopK': int(candidates_preview.groupby('sectionId').size().max()), 'unmappedRows': int(matches_preview['ncsSubCode'].isna().sum()), 'denseAllNull': bool(matches_preview['denseScore'].isna().all()), 'goldValidatedAny': bool(matches_preview['goldValidatedFlag'].any())}
assert input_audit['dutyRows'] == 28 and input_audit['maxTopK'] <= 5
assert input_audit['denseAllNull'] and not input_audit['goldValidatedAny']
input_audit

{'dutyRows': 28,
 'candidateRows': 128,
 'matchRows': 28,
 'maxTopK': 5,
 'unmappedRows': 1,
 'denseAllNull': True,
 'goldValidatedAny': False}

In [4]:
from p4_ncs.workflow.observed import run_stage

stage_manifest = run_stage('A4-03-MAP-OBSERVED', root=NCS_ROOT, duty_input_path=resolved_duty_input or None, gold_input_path=resolved_gold_input or None, schema_dir=resolved_schema_dir or None)
stage_manifest

{'manifestVersion': 'stage-manifest-v1',
 'runId': 'NCS_MAPPING_OBSERVED_20260806_01',
 'runMode': 'observed-dev',
 'stageId': 'A4-03-MAP-OBSERVED',
 'status': 'SUCCEEDED',
 'agentId': 'P4-A4-NCS',
 'branch': 'agent/p4-ncs-mapping-v2',
 'gitHead': 'cce6067e578cb8dc99aacaeb465439cb1ef0faa1',
 'contractVersion': '2.1.2',
 'schemaVersion': 'posting-ncs-candidates-v1',
 'dataVersion': 'observed-dev-20260806.1',
 'crawlReleaseId': 'CRAWL_20260806_03',
 'dataProvenance': 'OBSERVED_DEVELOPMENT_ONLY',
 'startedAt': '2026-08-06T08:38:46.033992Z',
 'completedAt': '2026-08-06T08:38:46.036069Z',
 'empiricalAnalysisAllowed': False,
 'promotionAllowed': False,
 'inputManifestSha256': 'a305d2354a7437860dc947f4e537be76c6233cd26fe658da650d3bb07437bc30',
 'parameterSha256': 'ee2a95b4cf722fbc15def7f7c955eba2248f681ec2a54ffef33e1869c3836e36',
 'rowCounts': {'dutyInput': 28,
  'postingNcsCandidates': 128,
  'postingNcsMatches': 28},
 'gateResults': [{'gateId': 'NCS_MAPPING_DEV_READY',
   'status': 'PASS',


In [5]:
stage_root = NCS_ROOT / 'data/runs' / RUN_MODE / 'NCS_MAPPING_OBSERVED_20260806_01' / stage_manifest['stageId']
expected_artifacts = {'stage_manifest.json', 'stage_metrics.json', 'stage_quality.csv', 'CHECKSUMS.sha256'}
actual_artifacts = {path.name for path in stage_root.iterdir() if path.is_file()}
assert actual_artifacts == expected_artifacts
termination_summary = {'stageId': stage_manifest['stageId'], 'status': stage_manifest['status'], 'rowCounts': stage_manifest['rowCounts'], 'artifacts': sorted(actual_artifacts)}
termination_summary

{'stageId': 'A4-03-MAP-OBSERVED',
 'status': 'SUCCEEDED',
 'rowCounts': {'dutyInput': 28,
  'postingNcsCandidates': 128,
  'postingNcsMatches': 28},
 'artifacts': ['CHECKSUMS.sha256',
  'stage_manifest.json',
  'stage_metrics.json',
  'stage_quality.csv']}